# 획득 함수 실습

**Acquisition Function · EI · UCB · 기대 개선**

예측값과 불확실성을 하나의 점수로 합쳐 다음에 평가할 조건을 정하는 함수.

소재 분야에서 이해하기: 기대 개선이 가장 큰 조성을 다음 실험으로 고른다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [베이지안 능동학습 연구](https://www.nature.com/articles/s41467-020-19597-w)

## 1. 같은 모델, 다른 선택 기준

EI·UCB·PI가 같은 상황에서 서로 다른 다음 실험을 고르는 것을 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from scipy.stats import norm

def objective(x):
    return np.sin(3 * x) + 0.3 * x

x_seen = np.array([0.4, 1.0, 1.6, 3.0])
y_seen = objective(x_seen)
grid = np.linspace(0, 4, 500)
model = GaussianProcessRegressor(kernel=ConstantKernel(1.0) * RBF(0.5),
                                 normalize_y=True, alpha=1e-6, random_state=0).fit(x_seen[:, None], y_seen)
mean, std = model.predict(grid[:, None], return_std=True)
best = y_seen.max()
print('현재 최고값 %.3f' % best)

In [ ]:
def ei(mean, std, best):
    std = np.maximum(std, 1e-9); z = (mean - best) / std
    return (mean - best) * norm.cdf(z) + std * norm.pdf(z)

def pi(mean, std, best):
    return norm.cdf((mean - best) / np.maximum(std, 1e-9))

def ucb(mean, std, kappa):
    return mean + kappa * std

acquisitions = {'EI': ei(mean, std, best), 'PI': pi(mean, std, best),
                'UCB (k=1)': ucb(mean, std, 1.0), 'UCB (k=4)': ucb(mean, std, 4.0)}

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axes[0].plot(grid, objective(grid), 'k--', label='truth')
axes[0].plot(grid, mean, label='GP mean')
axes[0].fill_between(grid, mean - 2 * std, mean + 2 * std, alpha=0.2)
axes[0].scatter(x_seen, y_seen, c='red', zorder=5); axes[0].legend(fontsize=8)
for name, values in acquisitions.items():
    axes[1].plot(grid, values / np.max(np.abs(values)), label=name)
    print('%-10s 가 고른 다음 실험 x=%.3f' % (name, grid[int(np.argmax(values))]))
axes[1].legend(fontsize=8); axes[1].set_xlabel('x'); axes[1].set_ylabel('normalised acquisition')
plt.tight_layout(); plt.show()

## 2. 탐색과 활용의 균형

In [ ]:
def optimise(acquisition, budget=12, seed=0):
    local = np.random.default_rng(seed)
    xs = list(local.uniform(0, 4, 3)); ys = [objective(v) for v in xs]
    for _ in range(budget):
        local_model = GaussianProcessRegressor(kernel=ConstantKernel(1.0) * RBF(0.5),
                                               normalize_y=True, alpha=1e-6, random_state=0)
        local_model.fit(np.array(xs)[:, None], ys)
        m, s = local_model.predict(grid[:, None], return_std=True)
        values = acquisition(m, s, max(ys))
        pick = grid[int(np.argmax(values))]
        xs.append(float(pick)); ys.append(float(objective(pick)))
    return max(ys), np.std(xs)

for name, acquisition in [('EI', ei), ('PI', pi),
                          ('UCB k=1', lambda m, s, b: ucb(m, s, 1.0)),
                          ('UCB k=5', lambda m, s, b: ucb(m, s, 5.0))]:
    best_value, spread = optimise(acquisition)
    print('%-9s 최종 최고값 %.3f, 실험 위치 표준편차 %.2f (클수록 넓게 탐색)' % (name, best_value, spread))
print('\n참 최댓값 %.3f' % objective(grid).max())
print('PI 는 이미 좋은 곳 근처만 보는 경향, UCB 의 k 를 키우면 탐색이 넓어집니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#acquisition-function)을 여세요.